# River-flood forest restoration encroachment risk

This notebook applies the mangrove encroachment-risk logic to river-flood forest restoration areas with positive avoided EAD. Because the restoration-benefit surface is a 30 m raster, the encroachment analysis is also run on that 30 m grid: land cover is rasterized to the EAD grid, 10 m is interpreted as direct adjacency to the restoration-benefit cell edge, and 100 m is interpreted using the same 30 m cell-geometry distance rule.

In [ ]:
from pathlib import Path
import re

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from IPython.display import Markdown, display
from rasterio.features import rasterize
from scipy.ndimage import binary_dilation

In [ ]:
BASE = Path("/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers")
PAPER2 = BASE / "dphil_paper_2"
PAPER3 = BASE / "dphil_paper_3"
COMMON = BASE / "dphil_common_cross_cutting"

OUT_DIR = PAPER3 / "results" / "threats" / "encroachment_risk" / "river_flood_restoration_encroachment"
OUT_DIR.mkdir(parents=True, exist_ok=True)

RIVER_EAD_MIN_PATH = PAPER2 / "processed_data" / "nbs_river_catchment" / "damage_reduction" / "damage_reduction_min.tif"
RIVER_EAD_MAX_PATH = PAPER2 / "processed_data" / "nbs_river_catchment" / "damage_reduction" / "damage_reduction_max.tif"
LAND_USE_PATH = COMMON / "common_incoming_data" / "landcover" / "2013_landcover" / "2013_landuse_LandCover.shp"

J2USD = 1 / 150
DISTANCES_M = [10, 100]
RASTERIZATION_RULE = "Land-cover classes rasterized to the 30 m EAD grid using pixel centres; distance rings use 30 m cell-geometry adjacency."

OUT_DIR

## Load positive avoided EAD rasters

The river-flood restoration rasters are converted from JMD to USD. Non-positive and missing values are treated as not providing positive avoided EAD.

In [ ]:
def read_positive_ead_usd(path: Path, reference: dict | None = None) -> tuple[np.ndarray, dict]:
    """Read avoided EAD, convert JMD to USD, and retain positive values only."""
    with rasterio.open(path) as src:
        profile = src.profile.copy()
        array = src.read(1).astype("float64") * J2USD

    array[~np.isfinite(array)] = np.nan
    array[array <= 0] = np.nan

    if reference is not None:
        checks = {
            "crs": profile["crs"] == reference["crs"],
            "transform": profile["transform"] == reference["transform"],
            "height": profile["height"] == reference["height"],
            "width": profile["width"] == reference["width"],
        }
        if not all(checks.values()):
            raise ValueError(f"Raster alignment mismatch for {path}: {checks}")

    return array, profile


river_ead_min_usd, raster_profile = read_positive_ead_usd(RIVER_EAD_MIN_PATH)
river_ead_max_usd, _ = read_positive_ead_usd(RIVER_EAD_MAX_PATH, raster_profile)

raster_shape = (raster_profile["height"], raster_profile["width"])
raster_transform = raster_profile["transform"]
raster_crs = raster_profile["crs"]
pixel_width_m = abs(raster_transform.a)
pixel_height_m = abs(raster_transform.e)
pixel_size_m = (pixel_width_m + pixel_height_m) / 2
pixel_area_ha = pixel_width_m * pixel_height_m / 10_000

minimum_positive_mask = np.isfinite(river_ead_min_usd) & (river_ead_min_usd > 0)
maximum_positive_mask = np.isfinite(river_ead_max_usd) & (river_ead_max_usd > 0)
benefit_mask = minimum_positive_mask | maximum_positive_mask

scenario_arrays = {
    "minimum": river_ead_min_usd,
    "maximum": river_ead_max_usd,
}
scenario_positive_masks = {
    "minimum": minimum_positive_mask,
    "maximum": maximum_positive_mask,
}
scenario_totals_usd = {
    scenario: float(np.nansum(array)) for scenario, array in scenario_arrays.items()
}

river_grid_summary = pd.DataFrame(
    [
        {
            "scenario": scenario,
            "positive_pixels": int(mask.sum()),
            "positive_area_ha": float(mask.sum() * pixel_area_ha),
            "positive_avoided_ead_usd": scenario_totals_usd[scenario],
            "positive_avoided_ead_usd_million": scenario_totals_usd[scenario] / 1e6,
        }
        for scenario, mask in scenario_positive_masks.items()
    ]
)

river_grid_summary

## Load and rasterize land cover

The encroachment-risk class grouping matches the mangrove analysis: buildings and other infrastructure, agriculture, bauxite extraction/quarry, and plantations.

In [ ]:
encroachment_classes = [
    "Buildings and other infrastructure",
    "Agriculture",
    "Bauxite extraction / quarry",
    "Plantation",
]

encroachment_class_map = {
    "Buildings and other infrastructure": [
        "Buildings and other infrastructures",
    ],
    "Agriculture": [
        "Fields: Herbaceous crops, fallow, cultivated vegetables",
        "Fields: Pasture,Human disturbed, grassland",
        "Fields: Bare Land",
        "Fields and Secondary Forest",
        "Fields or Secondary Forest/Pine Plantation",
        "Fields  and Bamboo",
        "Bamboo and Fields",
    ],
    "Bauxite extraction / quarry": [
        "Bauxite Extraction",
        "Quarry",
    ],
    "Plantation": [
        "Plantation: Tree crops, shrub crops, sugar cane, banana",
        "Hardwood Plantation: Euculytus",
        "Hardwood Plantation: Mahoe",
        "Hardwood Plantation: Mahogany",
        "Hardwood Plantation: Mixed",
    ],
}


def normalise_class_name(value: object) -> str:
    return re.sub(r"\s+", " ", str(value).strip().lower())


normalised_encroachment_lookup = {
    normalise_class_name(source_class): encroachment_class
    for encroachment_class, source_classes in encroachment_class_map.items()
    for source_class in source_classes
}


def assign_encroachment_class(classify_value: object) -> str | None:
    return normalised_encroachment_lookup.get(normalise_class_name(classify_value))


terrestrial_landcover = gpd.read_file(LAND_USE_PATH, columns=["Classify", "geometry"]).to_crs(raster_crs)
terrestrial_landcover = terrestrial_landcover[
    terrestrial_landcover.geometry.notna() & ~terrestrial_landcover.geometry.is_empty
].copy()
terrestrial_landcover["encroachment_class"] = terrestrial_landcover["Classify"].apply(assign_encroachment_class)

landcover_classes = sorted(terrestrial_landcover["Classify"].dropna().unique())
landcover_code_lookup = {landcover_class: code for code, landcover_class in enumerate(landcover_classes, start=1)}
landcover_label_lookup = {code: landcover_class for landcover_class, code in landcover_code_lookup.items()}

encroachment_code_lookup = {
    encroachment_class: code for code, encroachment_class in enumerate(encroachment_classes, start=1)
}
encroachment_label_lookup = {
    code: encroachment_class for encroachment_class, code in encroachment_code_lookup.items()
}

terrestrial_landcover["landcover_code"] = terrestrial_landcover["Classify"].map(landcover_code_lookup).astype("uint16")
terrestrial_landcover["encroachment_code"] = (
    terrestrial_landcover["encroachment_class"].map(encroachment_code_lookup).fillna(0).astype("uint8")
)

landcover_code_raster = rasterize(
    zip(terrestrial_landcover.geometry, terrestrial_landcover["landcover_code"]),
    out_shape=raster_shape,
    transform=raster_transform,
    fill=0,
    dtype="uint16",
    all_touched=False,
)
encroachment_code_raster = rasterize(
    zip(terrestrial_landcover.geometry, terrestrial_landcover["encroachment_code"]),
    out_shape=raster_shape,
    transform=raster_transform,
    fill=0,
    dtype="uint8",
    all_touched=False,
)

landcover_raster_summary = pd.DataFrame(
    [
        {
            "landcover_features": len(terrestrial_landcover),
            "landcover_classes": len(landcover_classes),
            "encroachment_risk_features": int(terrestrial_landcover["encroachment_class"].notna().sum()),
            "classified_pixels_in_ead_grid": int((landcover_code_raster > 0).sum()),
            "classified_area_ha_in_ead_grid": float((landcover_code_raster > 0).sum() * pixel_area_ha),
            "crs": str(terrestrial_landcover.crs),
        }
    ]
)

landcover_raster_summary

## Define 10 m and 100 m raster rings

The avoided-EAD grid is 30 m. A cell is included in a distance ring when the square cell geometry is within the requested distance of a restoration-benefit cell. This treats the 10 m ring as immediate edge/corner adjacency, because a 10 m distance is below the EAD pixel size.

In [ ]:
def cell_distance_structure(distance_m: int, pixel_size: float) -> np.ndarray:
    """Return a dilation footprint based on minimum distance between square cells."""
    max_offset = int(np.ceil(distance_m / pixel_size)) + 1
    row_offsets, col_offsets = np.ogrid[-max_offset : max_offset + 1, -max_offset : max_offset + 1]
    horizontal_gap_m = np.maximum(np.abs(col_offsets) - 1, 0) * pixel_size
    vertical_gap_m = np.maximum(np.abs(row_offsets) - 1, 0) * pixel_size
    cell_gap_m = np.sqrt(horizontal_gap_m**2 + vertical_gap_m**2)
    return cell_gap_m <= distance_m


def ring_from_mask(mask: np.ndarray, distance_m: int, pixel_size: float) -> tuple[np.ndarray, np.ndarray]:
    structure = cell_distance_structure(distance_m, pixel_size)
    expanded_mask = binary_dilation(mask, structure=structure)
    ring_mask = expanded_mask & ~mask
    return ring_mask, structure


ring_masks = {}
ring_structures = {}
ring_definition_rows = []

for distance_m in DISTANCES_M:
    ring_mask, structure = ring_from_mask(benefit_mask, distance_m, pixel_size_m)
    ring_masks[distance_m] = ring_mask
    ring_structures[distance_m] = structure
    ring_definition_rows.append(
        {
            "distance_m": distance_m,
            "footprint_rows": structure.shape[0],
            "footprint_cols": structure.shape[1],
            "included_offsets": int(structure.sum()),
            "ring_pixels": int(ring_mask.sum()),
            "ring_area_ha": float(ring_mask.sum() * pixel_area_ha),
        }
    )

ring_definition = pd.DataFrame(ring_definition_rows)
ring_definition

## Summarise land cover in the surrounding rings

In [ ]:
def summarise_landcover_codes(code_raster: np.ndarray, mask: np.ndarray, label_lookup: dict[int, str]) -> pd.DataFrame:
    selected_codes = code_raster[mask]
    selected_codes = selected_codes[selected_codes > 0]
    if selected_codes.size == 0:
        return pd.DataFrame(columns=["code", "label", "pixels", "area_ha"])

    counts = np.bincount(selected_codes.astype("int64"), minlength=max(label_lookup) + 1)
    rows = [
        {
            "code": code,
            "label": label_lookup[code],
            "pixels": int(count),
            "area_ha": float(count * pixel_area_ha),
        }
        for code, count in enumerate(counts)
        if code in label_lookup and count > 0
    ]
    return pd.DataFrame(rows).sort_values("area_ha", ascending=False)


ring_landcover_summary_parts = []
ring_encroachment_by_class_parts = []
ring_metadata_rows = []

for distance_m in DISTANCES_M:
    ring_mask = ring_masks[distance_m]
    landcover_summary = summarise_landcover_codes(landcover_code_raster, ring_mask, landcover_label_lookup)
    classified_area_ha = float(landcover_summary["area_ha"].sum()) if not landcover_summary.empty else 0.0

    if not landcover_summary.empty:
        landcover_summary["pct_of_classified_ring"] = landcover_summary["area_ha"] / classified_area_ha * 100
        landcover_summary["pct_of_full_ring"] = landcover_summary["area_ha"] / (ring_mask.sum() * pixel_area_ha) * 100
        landcover_summary["distance_m"] = distance_m
        landcover_summary["encroachment_class"] = landcover_summary["label"].apply(assign_encroachment_class)
        landcover_summary = landcover_summary.rename(columns={"label": "landcover_class"})

    encroachment_summary = summarise_landcover_codes(
        encroachment_code_raster,
        ring_mask,
        encroachment_label_lookup,
    )
    encroachment_area_ha = float(encroachment_summary["area_ha"].sum()) if not encroachment_summary.empty else 0.0

    if not encroachment_summary.empty:
        encroachment_summary["pct_of_classified_ring"] = encroachment_summary["area_ha"] / classified_area_ha * 100
        encroachment_summary["pct_of_full_ring"] = encroachment_summary["area_ha"] / (ring_mask.sum() * pixel_area_ha) * 100
        encroachment_summary["distance_m"] = distance_m
        encroachment_summary = encroachment_summary.rename(columns={"label": "encroachment_class"})

    ring_landcover_summary_parts.append(landcover_summary)
    ring_encroachment_by_class_parts.append(encroachment_summary)
    ring_metadata_rows.append(
        {
            "distance_m": distance_m,
            "ring_area_ha": float(ring_mask.sum() * pixel_area_ha),
            "classified_ring_area_ha": classified_area_ha,
            "classified_pct_of_full_ring": classified_area_ha / (ring_mask.sum() * pixel_area_ha) * 100 if ring_mask.sum() else 0.0,
            "encroachment_area_ha": encroachment_area_ha,
            "encroachment_pct_of_classified_ring": encroachment_area_ha / classified_area_ha * 100 if classified_area_ha else 0.0,
            "encroachment_pct_of_full_ring": encroachment_area_ha / (ring_mask.sum() * pixel_area_ha) * 100 if ring_mask.sum() else 0.0,
        }
    )

ring_landcover_summary = pd.concat(ring_landcover_summary_parts, ignore_index=True)
ring_encroachment_by_class = pd.concat(ring_encroachment_by_class_parts, ignore_index=True)
ring_metadata = pd.DataFrame(ring_metadata_rows)

ring_metadata

In [ ]:
ring_landcover_summary_out = OUT_DIR / "river_flood_restoration_ring_landcover_summary.csv"
ring_encroachment_by_class_out = OUT_DIR / "river_flood_restoration_ring_encroachment_by_class.csv"
ring_metadata_out = OUT_DIR / "river_flood_restoration_ring_metadata.csv"
other_classified_landcover_out = OUT_DIR / "river_flood_restoration_other_classified_landcover.csv"
ring_definition_out = OUT_DIR / "river_flood_restoration_ring_definition.csv"
landcover_raster_summary_out = OUT_DIR / "river_flood_restoration_landcover_raster_summary.csv"

ring_landcover_summary.to_csv(ring_landcover_summary_out, index=False)
ring_encroachment_by_class.to_csv(ring_encroachment_by_class_out, index=False)
ring_metadata.to_csv(ring_metadata_out, index=False)
other_classified_landcover = ring_landcover_summary[ring_landcover_summary["encroachment_class"].isna()].copy()
other_classified_landcover.to_csv(other_classified_landcover_out, index=False)
ring_definition.to_csv(ring_definition_out, index=False)
landcover_raster_summary.to_csv(landcover_raster_summary_out, index=False)

print(f"Saved: {ring_landcover_summary_out}")
print(f"Saved: {ring_encroachment_by_class_out}")
print(f"Saved: {ring_metadata_out}")
print(f"Saved: {other_classified_landcover_out}")
print(f"Saved: {ring_definition_out}")
print(f"Saved: {landcover_raster_summary_out}")

display(ring_encroachment_by_class)

## Estimate restoration-benefit area and avoided EAD exposed to external encroachment-risk land use

Only encroachment-risk land-cover cells in the external ring are used. Benefit pixels are then flagged when they are within the same distance of those external risk cells. This avoids treating encroachment-risk land-cover classes inside the restoration-benefit area itself as surrounding pressure.

In [ ]:
def ead_sum_usd(ead_array: np.ndarray, mask: np.ndarray) -> float:
    return float(np.nansum(ead_array[mask]))


def pct(part: float, whole: float) -> float:
    return part / whole * 100 if whole else 0.0


exposure_overall_rows = []
exposure_scenario_rows = []
exposure_masks = {}

for distance_m in DISTANCES_M:
    ring_mask = ring_masks[distance_m]
    external_encroachment_mask = ring_mask & (encroachment_code_raster > 0)
    exposed_mask = binary_dilation(
        external_encroachment_mask,
        structure=ring_structures[distance_m],
    ) & benefit_mask
    exposure_masks[distance_m] = exposed_mask

    exposed_area_ha = float(exposed_mask.sum() * pixel_area_ha)
    benefit_area_ha = float(benefit_mask.sum() * pixel_area_ha)

    exposure_overall_rows.append(
        {
            "distance_m": distance_m,
            "benefit_area_ha_union_min_or_max": benefit_area_ha,
            "exposed_benefit_pixels": int(exposed_mask.sum()),
            "exposed_benefit_area_ha": exposed_area_ha,
            "pct_benefit_area_exposed": pct(exposed_area_ha, benefit_area_ha),
            "not_exposed_benefit_area_ha": benefit_area_ha - exposed_area_ha,
            "external_encroachment_pixels_in_ring": int(external_encroachment_mask.sum()),
            "external_encroachment_area_ha_in_ring": float(external_encroachment_mask.sum() * pixel_area_ha),
            "rasterization_rule": RASTERIZATION_RULE,
        }
    )

    for scenario, positive_mask in scenario_positive_masks.items():
        scenario_exposed_mask = exposed_mask & positive_mask
        scenario_area_ha = float(positive_mask.sum() * pixel_area_ha)
        exposed_scenario_area_ha = float(scenario_exposed_mask.sum() * pixel_area_ha)
        exposed_ead_usd = ead_sum_usd(scenario_arrays[scenario], scenario_exposed_mask)
        total_ead_usd = scenario_totals_usd[scenario]

        exposure_scenario_rows.append(
            {
                "distance_m": distance_m,
                "scenario": scenario,
                "positive_area_ha": scenario_area_ha,
                "exposed_positive_area_ha": exposed_scenario_area_ha,
                "pct_positive_area_exposed": pct(exposed_scenario_area_ha, scenario_area_ha),
                "positive_avoided_ead_usd": total_ead_usd,
                "exposed_positive_avoided_ead_usd": exposed_ead_usd,
                "exposed_positive_avoided_ead_usd_million": exposed_ead_usd / 1e6,
                "pct_positive_avoided_ead_exposed": pct(exposed_ead_usd, total_ead_usd),
                "not_exposed_positive_avoided_ead_usd": total_ead_usd - exposed_ead_usd,
            }
        )

exposure_overall_summary = pd.DataFrame(exposure_overall_rows)
exposure_scenario_summary = pd.DataFrame(exposure_scenario_rows)

exposure_overall_summary

In [ ]:
exposure_overall_out = OUT_DIR / "river_flood_restoration_encroachment_exposure_overall_summary.csv"
exposure_scenario_out = OUT_DIR / "river_flood_restoration_encroachment_exposure_scenario_summary.csv"
river_grid_summary_out = OUT_DIR / "river_flood_restoration_encroachment_grid_summary.csv"

exposure_overall_summary.to_csv(exposure_overall_out, index=False)
exposure_scenario_summary.to_csv(exposure_scenario_out, index=False)
river_grid_summary.to_csv(river_grid_summary_out, index=False)

print(f"Saved: {exposure_overall_out}")
print(f"Saved: {exposure_scenario_out}")
print(f"Saved: {river_grid_summary_out}")

display(exposure_scenario_summary)

## Optional figure: composition of encroachment-risk land use in surrounding rings

A map is not very informative for this analysis because the relevant feature is a narrow ring around a very extensive raster surface. This compact stacked bar chart is more useful if a supplementary figure is needed.

In [ ]:
category_colors = {
    "Buildings and other infrastructure": "#5f6368",
    "Agriculture": "#d8a53a",
    "Bauxite extraction / quarry": "#b76534",
    "Plantation": "#3f8f62",
    "Natural land cover, water, bamboo & bare rock": "#d9d9d9",
}
category_order = encroachment_classes + ["Natural land cover, water, bamboo & bare rock"]
composition_rows = []

for distance_m in DISTANCES_M:
    distance_rows = ring_encroachment_by_class[ring_encroachment_by_class["distance_m"] == distance_m]
    risk_pct_lookup = dict(zip(distance_rows["encroachment_class"], distance_rows["pct_of_classified_ring"]))
    total_risk_pct = sum(risk_pct_lookup.values())
    for category in encroachment_classes:
        composition_rows.append(
            {
                "distance_m": distance_m,
                "category": category,
                "pct_of_classified_ring": risk_pct_lookup.get(category, 0.0),
            }
        )
    composition_rows.append(
        {
            "distance_m": distance_m,
            "category": "Natural land cover, water, bamboo & bare rock",
            "pct_of_classified_ring": max(0.0, 100.0 - total_risk_pct),
        }
    )

ring_composition = pd.DataFrame(composition_rows)

with plt.rc_context({"font.size": 9, "axes.labelsize": 9, "xtick.labelsize": 8, "ytick.labelsize": 8}):
    figure, axis = plt.subplots(figsize=(6.4, 3.6))
    distance_labels = [f"{distance_m} m" for distance_m in DISTANCES_M]
    bottom = np.zeros(len(DISTANCES_M))

    for category in category_order:
        values = [
            ring_composition.loc[
                (ring_composition["distance_m"] == distance_m)
                & (ring_composition["category"] == category),
                "pct_of_classified_ring",
            ].sum()
            for distance_m in DISTANCES_M
        ]
        axis.bar(
            distance_labels,
            values,
            bottom=bottom,
            color=category_colors[category],
            edgecolor="white",
            linewidth=0.5,
            label=category,
        )
        bottom += np.array(values)

    axis.set_ylabel("Share of classified\nsurrounding land cover (%)", labelpad=8)
    axis.set_xlabel("Distance from restoration-benefit area", labelpad=6)
    axis.set_ylim(0, 100)
    axis.spines[["top", "right"]].set_visible(False)
    axis.legend(
        loc="upper center",
        bbox_to_anchor=(0.5, -0.24),
        ncol=2,
        frameon=False,
        fontsize=7.5,
        columnspacing=1.2,
        handlelength=1.4,
    )
    figure.subplots_adjust(left=0.18, right=0.98, bottom=0.36, top=0.96)

    figure_paths = []
    for extension in ["png", "pdf", "svg"]:
        figure_path = OUT_DIR / f"river_flood_restoration_encroachment_ring_composition.{extension}"
        figure.savefig(figure_path, dpi=300)
        figure_paths.append(figure_path)

    plt.show()

figure_paths

## Draft results text

In [ ]:
def fnum(value: float, digits: int = 1) -> str:
    return f"{value:,.{digits}f}"


def fpct(value: float, digits: int = 1) -> str:
    return f"{value:.{digits}f}%"


def usd_million_range(minimum_value: float, maximum_value: float) -> str:
    return f"US${minimum_value / 1e6:,.2f}-{maximum_value / 1e6:,.2f} million"


def scenario_value(distance_m: int, scenario: str, column: str) -> float:
    return float(
        exposure_scenario_summary.loc[
            (exposure_scenario_summary["distance_m"] == distance_m)
            & (exposure_scenario_summary["scenario"] == scenario),
            column,
        ].iloc[0]
    )


def metadata_value(distance_m: int, column: str) -> float:
    return float(ring_metadata.loc[ring_metadata["distance_m"] == distance_m, column].iloc[0])


def overall_value(distance_m: int, column: str) -> float:
    return float(exposure_overall_summary.loc[exposure_overall_summary["distance_m"] == distance_m, column].iloc[0])


ten_m_area = overall_value(10, "exposed_benefit_area_ha")
ten_m_area_pct = overall_value(10, "pct_benefit_area_exposed")
ten_m_ead_min = scenario_value(10, "minimum", "exposed_positive_avoided_ead_usd")
ten_m_ead_max = scenario_value(10, "maximum", "exposed_positive_avoided_ead_usd")
ten_m_ead_pct_min = scenario_value(10, "minimum", "pct_positive_avoided_ead_exposed")
ten_m_ead_pct_max = scenario_value(10, "maximum", "pct_positive_avoided_ead_exposed")

hundred_m_area = overall_value(100, "exposed_benefit_area_ha")
hundred_m_area_pct = overall_value(100, "pct_benefit_area_exposed")
hundred_m_ead_min = scenario_value(100, "minimum", "exposed_positive_avoided_ead_usd")
hundred_m_ead_max = scenario_value(100, "maximum", "exposed_positive_avoided_ead_usd")
hundred_m_ead_pct_min = scenario_value(100, "minimum", "pct_positive_avoided_ead_exposed")
hundred_m_ead_pct_max = scenario_value(100, "maximum", "pct_positive_avoided_ead_exposed")
benefit_area_ha = overall_value(10, "benefit_area_ha_union_min_or_max")

summary_text = f"""# River-flood forest restoration encroachment risk

Encroachment-risk land use accounts for {fpct(metadata_value(10, 'encroachment_pct_of_classified_ring'))} of classified land cover in the 10 m edge-adjacent raster ring around river-flood forest restoration areas with positive avoided EAD, compared with {fpct(metadata_value(100, 'encroachment_pct_of_classified_ring'))} in the wider 100 m surrounding raster ring.

Using the original 30 m avoided-EAD grid, {fnum(ten_m_area)} ha of the {fnum(benefit_area_ha)} ha of restoration-benefit area, or {fpct(ten_m_area_pct)}, is exposed to external encroachment-risk land use in the 10 m edge-adjacent ring. These exposed restoration-benefit pixels account for {usd_million_range(ten_m_ead_min, ten_m_ead_max)} in positive avoided EAD, equivalent to {fpct(ten_m_ead_pct_min)} and {fpct(ten_m_ead_pct_max)} of total positive river-flood restoration avoided EAD in the minimum and maximum scenarios, respectively.

Within the 100 m surrounding raster ring, {fnum(hundred_m_area)} ha of restoration-benefit area, or {fpct(hundred_m_area_pct)}, is exposed to surrounding encroachment-risk land use. This wider exposure area accounts for {usd_million_range(hundred_m_ead_min, hundred_m_ead_max)} in positive avoided EAD, equivalent to {fpct(hundred_m_ead_pct_min)} and {fpct(hundred_m_ead_pct_max)} of total positive river-flood restoration avoided EAD.
"""

summary_path = OUT_DIR / "river_flood_restoration_encroachment_written_summary.md"
summary_path.write_text(summary_text)

display(Markdown(summary_text))
summary_path

## Write method metadata

In [ ]:
method_metadata = pd.DataFrame(
    [
        {"name": "river_ead_min_path", "value": str(RIVER_EAD_MIN_PATH)},
        {"name": "river_ead_max_path", "value": str(RIVER_EAD_MAX_PATH)},
        {"name": "land_use_path", "value": str(LAND_USE_PATH)},
        {"name": "benefit_area_definition", "value": "positive avoided EAD in minimum or maximum river-flood restoration scenario"},
        {"name": "encroachment_classes", "value": "; ".join(encroachment_classes)},
        {"name": "ring_distances_m", "value": "; ".join(str(distance_m) for distance_m in DISTANCES_M)},
        {"name": "currency_conversion", "value": "J2USD = 1 / 150"},
        {"name": "rasterization_rule", "value": RASTERIZATION_RULE},
    ]
)

method_metadata_out = OUT_DIR / "river_flood_restoration_encroachment_method_metadata.csv"
method_metadata.to_csv(method_metadata_out, index=False)

[
    river_grid_summary_out,
    landcover_raster_summary_out,
    ring_definition_out,
    ring_landcover_summary_out,
    ring_encroachment_by_class_out,
    ring_metadata_out,
    other_classified_landcover_out,
    exposure_overall_out,
    exposure_scenario_out,
    summary_path,
    method_metadata_out,
    *figure_paths,
]